# 21 — 현재 청킹 분기 전 reranker 종합 평가

이 노트북은 **Codex coder agent**가 `skn25`, `CUDA_VISIBLE_DEVICES=0`, local-only 조건에서 실행한다. Phase A는 질문 문구만 보는 실제 regex route를 저장된 18번 순위로 평가하고, Phase B는 현재 section/benefit 후보에서 GTE/BGE × 원문/경로보강 2×2를 같은 933쌍에 실제 재점수화한다.

정답 유형을 미리 아는 selective oracle은 상한 진단일 뿐 운영 router가 아니다. actual regex route만 query text를 입력으로 받는다. 모든 결과는 개발셋 단일 실행이며 promotion·holdout·운영 일반화 근거가 아니다.


In [1]:
import os
from pathlib import Path
PROJECT_ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
os.environ['HF_HOME']=str((PROJECT_ROOT/'.cache/huggingface').resolve()); os.environ['HF_HUB_OFFLINE']='1'; os.environ['TRANSFORMERS_OFFLINE']='1'; os.environ['HF_HUB_DISABLE_TELEMETRY']='1'; os.environ['TOKENIZERS_PARALLELISM']='false'
assert os.environ.get('CUDA_VISIBLE_DEVICES')=='0'
import csv, gc, hashlib, importlib.metadata, itertools, json, math, re, statistics, tempfile, time, unicodedata
from collections import Counter
import numpy as np
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
OUTPUT_ROOT=PROJECT_ROOT/'notebooks/data/21_current_chunking_prebranch_reranker_evaluation'; OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
S13=PROJECT_ROOT/'notebooks/data/13_hierarchical_chunking_retrieval'; S16=PROJECT_ROOT/'notebooks/data/16_normalized_rrf_weight_ablation'; S17=PROJECT_ROOT/'notebooks/data/17_gte_reranker_ablation'; S18=PROJECT_ROOT/'notebooks/data/18_answer_bearing_grouped_retrieval_reevaluation'
GTE_ROOT=PROJECT_ROOT/'.cache/reranker/gte-multilingual-reranker-base'; BGE_ROOT=PROJECT_ROOT/'.cache/reranker/bge-reranker-v2-m3'; CUSTOM_HUB=PROJECT_ROOT/'.cache/huggingface/hub/models--Alibaba-NLP--new-impl/snapshots/40ced75c3017eb27626c9d4ea981bde21a2662f4'; CUSTOM_MODULE=PROJECT_ROOT/'.cache/huggingface/modules/transformers_modules/Alibaba_hyphen_NLP/new_hyphen_impl/40ced75c3017eb27626c9d4ea981bde21a2662f4'
INPUTS={'chunks_13':S13/'chunks.jsonl','queries_13':S13/'retrieval_per_query.csv','candidates_16':S16/'rrf_weight_candidates.csv','gte_scores_17':S17/'gte_reranker_pair_scores.csv','gte_per_query_17':S17/'gte_reranker_per_query.csv','gte_summary_17':S17/'gte_reranker_summary.json','contract_18':S18/'evaluation_contract.json','relevance_18':S18/'relevance_sets.jsonl','exact_groups_18':S18/'exact_document_groups.jsonl','per_query_18':S18/'per_query_metrics.csv','summary_18':S18/'summary.csv','leaf_per_query_18':S18/'leaf_only_per_query_metrics.csv','leaf_summary_18':S18/'leaf_only_summary.json','integrity_17':S17/'gte_reranker_run_manifest.json','integrity_18':S18/'leaf_only_integrity.json'}
WEIGHTS=('vector_0.4_bm25_0.6','vector_0.5_bm25_0.5'); DEPTHS=(20,50); VARIANTS=('leaf_only_top20','leaf_available_from_original_top50'); VIEWS=('strict_raw','answer_bearing_raw','strict_exact_doc_dedup','answer_bearing_exact_doc_dedup','answer_bearing_gold_family_oracle'); METRICS=('card_hit_at_3','hit_at_3','recall_at_5','mrr_at_5','ndcg_at_5'); GROUPS=('all','card','evidence','numeric','semantic'); DENOMS={'all':30,'card':10,'evidence':20,'numeric':10,'semantic':10}; EVIDENCE_GROUPS=('evidence','numeric','semantic'); TOL=1e-12; MAX_LENGTH=8192; BATCH_SIZE=2; GTE_REPRO_TOL=0.002
GTE_REV='8215cf04918ba6f7b6a62bb44238ce2953d8831c'; CUSTOM_REV='40ced75c3017eb27626c9d4ea981bde21a2662f4'; BGE_REV='953dc6f6f85a1b2dbfca4c34a2796e7dde08d41e'
EXPECTED_SHA={'gte_config':'995730781d157e147c13ccdfe0eb20a0875c486b6c4de8c97f0bbd845549dbc0','gte_tokenizer_config':'6f00514620aff01ba8b7291b2394e98daca5be264cb743805232d9ae27494b2a','bge_config':'13dcd6c31d9fec9d1d8e158702072f62d7fa7d312a64b9fe057bec9a08cfe41a','bge_tokenizer_config':'7e4c1cc848840aeccdd763458c18dd525eb0f795c992e00ebe9c28554e7db2d4','bge_model':'d9e3e081faff1eefb84019509b2f5558fd74c1a05a2c7db22f74174fcedb5286'}
PACKAGE_EXPECTED={'transformers':'4.57.6','tokenizers':'0.22.2','huggingface-hub':'0.36.2','torch':'2.11.0','safetensors':'0.8.0','sentencepiece':'0.2.2','einops':'0.8.2','setuptools':'83.0.0'}
RULE_TEXT='''normalization=NFKC/lower/whitespace/terminal punctuation; precedence=proper>numeric>semantic; proper=identity intent for issuer/company/bank/card product; numeric_direct=얼마|몇 unit|얼마나 discount/accrual; numeric_target=end targets discount/accrual rate, annual fee, fee, amount/limit, monthly/annual limit, per-liter discount, mileage/point accrual criterion, spend amount/criterion, use count/period; standalone 이용료 excluded; 연회비 면제 조건 semantic; raw-digit-only disabled'''; RULE_SHA=hashlib.sha256(RULE_TEXT.encode()).hexdigest()
CONTRACT={'schema_version':'current_chunking_prebranch_v1','provenance':'Codex coder agent','phase_a':{'classifier_rule_text':RULE_TEXT,'classifier_rule_sha256':RULE_SHA,'runtime_input_fields':['query_text'],'precedence':['proper','numeric','semantic_fallback'],'configs':['0.4/0.5 x Top20/Top50'],'systems':['all_no','all_gte','gold_oracle','actual_regex'],'final_status':['development_rule_only','not_validated_for_unseen_queries','not_eligible_for_promotion']},'phase_b':{'weights':list(WEIGHTS),'variants':list(VARIANTS),'levels':['section','benefit'],'expected_unique_pairs':933,'systems':['no_reranker','gte_original','gte_augmented','bge_original','bge_augmented'],'max_length':MAX_LENGTH,'truncation':'only_second','padding':'dynamic','dtype':'float16','batch_size':BATCH_SIZE,'ranking':'raw single logit desc; tie fused rank then chunk_id; no instruction/sigmoid/score mixing','primary':'0.4 + leaf_available_from_original_top50 + strict_exact_doc_dedup MRR@5','gte_saved_reproduction':{'diagnostic_logit_abs_tolerance':GTE_REPRO_TOL,'quality_hard_assertions':['finite 933/933','full rank exact 80/80 pools','Top5 exact 80/80 pools','18 leaf metrics exact at 1e-12'],'rationale':'pre-run diagnostic: fp16 dynamic-padding batch composition changed for 213/933 pairs; median/p95/p99 abs diff 0 and observed max 0.001708984375'},'promotion_eligible':False},'models':{'gte':{'path':'.cache/reranker/gte-multilingual-reranker-base','revision':GTE_REV,'custom_code_revision':CUSTOM_REV,'trust_remote_code':True,'config_sha256':EXPECTED_SHA['gte_config'],'tokenizer_config_sha256':EXPECTED_SHA['gte_tokenizer_config']},'bge':{'path':'.cache/reranker/bge-reranker-v2-m3','revision':BGE_REV,'architecture':'XLMRobertaForSequenceClassification','trust_remote_code':False,'config_max_position_embeddings':8194,'tokenizer_model_max_length':8192,'config_sha256':EXPECTED_SHA['bge_config'],'tokenizer_config_sha256':EXPECTED_SHA['bge_tokenizer_config'],'model_sha256':EXPECTED_SHA['bge_model']}},'execution':{'environment':'skn25','cuda_visible_devices':'0','physical_gpu':0,'network_api_calls':0,'new_embeddings':0,'chroma_queries':0,'package_installs':0,'pip_check_preexisting_warning':'torch 2.11.0 requires setuptools<82; installed setuptools 83.0.0; recorded only'}}
def sha(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()
def files_under(root):
    result=[]
    for directory,dirs,names in os.walk(root): dirs.sort(); result.extend(Path(directory)/name for name in sorted(names))
    return result
def tree_snapshot(root):
    files={p.relative_to(root).as_posix():{'sha256':sha(p),'bytes':p.stat().st_size} for p in files_under(root)}; h=hashlib.sha256()
    for rel,item in files.items(): h.update(rel.encode()); h.update(bytes.fromhex(item['sha256']))
    return {'file_count':len(files),'total_bytes':sum(v['bytes'] for v in files.values()),'tree_sha256':h.hexdigest(),'files':files}
def read_csv(path): return list(csv.DictReader(path.open(encoding='utf-8',newline='')))
def read_jsonl(path): return [json.loads(x) for x in path.read_text(encoding='utf-8').splitlines()]
def write_json(path,value):
    with tempfile.NamedTemporaryFile('w',encoding='utf-8',dir=path.parent,delete=False) as f: temp=Path(f.name); json.dump(value,f,ensure_ascii=False,indent=2); f.write('\n')
    os.replace(temp,path)
def write_csv(path,rows):
    rows=list(rows); fields=list(dict.fromkeys(k for r in rows for k in r))
    with tempfile.NamedTemporaryFile('w',encoding='utf-8',newline='',dir=path.parent,delete=False) as f: temp=Path(f.name); w=csv.DictWriter(f,fieldnames=fields); w.writeheader(); w.writerows(rows)
    os.replace(temp,path)
def norm(value): return ' '.join(unicodedata.normalize('NFKC',str(value)).lower().split())
def percentile(values,p):
    a=sorted(values); x=(len(a)-1)*p/100; lo=int(x); hi=min(lo+1,len(a)-1); return a[lo]+(a[hi]-a[lo])*(x-lo)
versions={name:importlib.metadata.version(name) for name in PACKAGE_EXPECTED}; assert versions==PACKAGE_EXPECTED
assert torch.cuda.is_available() and torch.cuda.device_count()==1 and torch.cuda.get_device_name(0).endswith('RTX 3090')
assert (GTE_ROOT/'.cache/huggingface/download/config.json.metadata').read_text().splitlines()[0]==GTE_REV and (BGE_ROOT/'.cache/huggingface/download/config.json.metadata').read_text().splitlines()[0]==BGE_REV
assert sha(GTE_ROOT/'config.json')==EXPECTED_SHA['gte_config'] and sha(GTE_ROOT/'tokenizer_config.json')==EXPECTED_SHA['gte_tokenizer_config'] and sha(BGE_ROOT/'config.json')==EXPECTED_SHA['bge_config'] and sha(BGE_ROOT/'tokenizer_config.json')==EXPECTED_SHA['bge_tokenizer_config'] and sha(BGE_ROOT/'model.safetensors')==EXPECTED_SHA['bge_model']
bge_config=json.loads((BGE_ROOT/'config.json').read_text()); bge_tok_config=json.loads((BGE_ROOT/'tokenizer_config.json').read_text()); assert bge_config['architectures']==['XLMRobertaForSequenceClassification'] and bge_config['max_position_embeddings']==8194 and bge_tok_config['model_max_length']==8192
source_before={name:sha(path) for name,path in INPUTS.items()}; cache_before={name:tree_snapshot(path) for name,path in {'gte':GTE_ROOT,'bge':BGE_ROOT,'custom_hub':CUSTOM_HUB,'custom_module':CUSTOM_MODULE}.items()}; write_json(OUTPUT_ROOT/'evaluation_contract.json',CONTRACT)
print({'contract_frozen':True,'gpu':torch.cuda.get_device_name(0),'versions':versions,'rule_sha':RULE_SHA,'cache_bytes':{k:v['total_bytes'] for k,v in cache_before.items()}})


/home/sms/anaconda3/envs/skn25/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'contract_frozen': True, 'gpu': 'NVIDIA GeForce RTX 3090', 'versions': {'transformers': '4.57.6', 'tokenizers': '0.22.2', 'huggingface-hub': '0.36.2', 'torch': '2.11.0', 'safetensors': '0.8.0', 'sentencepiece': '0.2.2', 'einops': '0.8.2', 'setuptools': '83.0.0'}, 'rule_sha': 'e4e2c0b33dc228382e269a3edb399b1b5b4d0c4f8d41149e96d4d06d64d55909', 'cache_bytes': {'gte': 629269531, 'bge': 2293568873, 'custom_hub': 66150, 'custom_module': 141830}}


## Phase A — 실제 regex 질의 분류기

분류 함수에는 query text 하나만 전달한다. gold category와 기대 카드·level·필수 용어는 분류가 끝난 뒤 평가에만 사용한다.


In [2]:
PROPER_RULES=(('proper_issuer_product',re.compile(r'(?:어느|어떤)\s*(?:(?:카드사|은행|회사)\s*)?상품(?:인가)?$')),('proper_issuer_direct',re.compile(r'(?:어느\s*)?(?:카드사|은행|회사)(?:인가)?$')),('proper_issuer_noun',re.compile(r'(?:발급사|발행사)(?:는|은|가|인가)?$')),('proper_where_action',re.compile(r'어디서\s*(?:발급|발행|출시)')) )
NUMERIC_RULES=(('numeric_direct_amount',re.compile(r'얼마(?:인가|나)?$')),('numeric_direct_count',re.compile(r'몇\s*(?:원|%|퍼센트|마일|마일리지|포인트|회|개월|일|년)(?:인가)?$')),('numeric_direct_how_much',re.compile(r'얼마나\s*(?:할인|적립|차감|청구)')),('numeric_target_end',re.compile(r'(?:할인율|적립률|연회비|수수료|(?:할인|적립)\s*(?:금액|한도)|(?:월|연간)\s*(?:할인|적립)?\s*한도|리터당\s*할인\s*금액|(?:마일리지|포인트)\s*적립\s*기준|실적\s*(?:금액|기준)|이용\s*(?:횟수|기간))(?:은|는|이|가|인가)?$')) )
def classify_query_text(query_text):
    text=' '.join(unicodedata.normalize('NFKC',str(query_text)).lower().split()).rstrip(' ?.!,。？！')
    for rule_id,pattern in PROPER_RULES:
        match=pattern.search(text)
        if match: return {'predicted_category':'proper_noun','matched_rule_id':rule_id,'matched_span':match.group(0),'normalized_query':text}
    if '연회비 면제 조건' not in text:
        for rule_id,pattern in NUMERIC_RULES:
            match=pattern.search(text)
            if match: return {'predicted_category':'numeric_condition','matched_rule_id':rule_id,'matched_span':match.group(0),'normalized_query':text}
    return {'predicted_category':'semantic','matched_rule_id':'semantic_fallback','matched_span':'','normalized_query':text}
sanity_specs=[('4대 주유소 혜택','semantic'),('1위 업종 적립률','numeric_condition'),('월 이용료를 돌려받는 혜택','semantic'),('항공권 결제 금액 차감 방법','semantic'),('IBK포인트3.8 어느 은행','proper_noun'),('발급사','proper_noun'),('연회비','numeric_condition'),('마일리지 어떻게 적립','semantic'),('리터당 할인금액','numeric_condition'),('월 할인한도','numeric_condition'),('연회비 면제조건','semantic'),('연회비 얼마','numeric_condition')]
sanity_rows=[]
for text,expected in sanity_specs:
    result=classify_query_text(text); sanity_rows.append({'query_text':text,'expected_category':expected,**result,'passed':result['predicted_category']==expected})
assert all(row['passed'] for row in sanity_rows)
raw_queries=read_csv(INPUTS['queries_13']); query_gold={}
for row in raw_queries:
    if row['method']=='keyword': query_gold[row['query_id']]={'query_id':row['query_id'],'query':row['query'],'category':row['category'],'expected_card':row['expected_card'],'expected_level':row['expected_level'],'required_terms':json.loads(row['required_terms'])}
assert len(query_gold)==30 and Counter(v['category'] for v in query_gold.values())==Counter({'proper_noun':10,'numeric_condition':10,'semantic':10})
classification=[]
for query_id,item in query_gold.items():
    result=classify_query_text(item['query']); classification.append({'query_id':query_id,'gold_category':item['category'],**result,'correct':result['predicted_category']==item['category'],'actual_rerank':result['predicted_category']=='semantic','gold_rerank':item['category']=='semantic'})
labels=('proper_noun','numeric_condition','semantic'); confusion={gold:{pred:sum(r['gold_category']==gold and r['predicted_category']==pred for r in classification) for pred in labels} for gold in labels}
class_metrics={}
for label in labels:
    tp=confusion[label][label]; fp=sum(confusion[g][label] for g in labels if g!=label); fn=sum(confusion[label][p] for p in labels if p!=label); support=sum(confusion[label].values()); precision=tp/(tp+fp) if tp+fp else 0; recall=tp/(tp+fn) if tp+fn else 0; f1=2*precision*recall/(precision+recall) if precision+recall else 0; class_metrics[label]={'precision':precision,'recall':recall,'f1':f1,'support':support}
accuracy=sum(r['correct'] for r in classification)/30; macro={m:statistics.fmean(class_metrics[x][m] for x in labels) for m in ('precision','recall','f1')}; weighted={m:sum(class_metrics[x][m]*class_metrics[x]['support'] for x in labels)/30 for m in ('precision','recall','f1')}; balanced=statistics.fmean(class_metrics[x]['recall'] for x in labels)
tp=sum(r['actual_rerank'] and r['gold_rerank'] for r in classification); tn=sum(not r['actual_rerank'] and not r['gold_rerank'] for r in classification); fp=sum(r['actual_rerank'] and not r['gold_rerank'] for r in classification); fn=sum(not r['actual_rerank'] and r['gold_rerank'] for r in classification); binary={'tp':tp,'tn':tn,'fp':fp,'fn':fn,'precision':tp/(tp+fp) if tp+fp else 0,'recall':tp/(tp+fn) if tp+fn else 0,'f1':2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0,'specificity':tn/(tn+fp) if tn+fp else 0,'false_reranker':fp,'missed_reranker':fn}
classification_summary={'rule_text':RULE_TEXT,'rule_sha256':RULE_SHA,'confusion':confusion,'accuracy':accuracy,'class_metrics':class_metrics,'macro':macro,'weighted':weighted,'balanced_accuracy':balanced,'semantic_rerank_binary':binary,'decision':'dev_rule_exactly_reproduces_gold_route' if accuracy==1 else 'dev_rule_imperfect_does_not_reproduce_gold_route','final_status':['development_rule_only','not_validated_for_unseen_queries','not_eligible_for_promotion']}
prediction={r['query_id']:r['predicted_category'] for r in classification}; assert len(prediction)==30
print({'classification_accuracy':accuracy,'confusion':confusion,'binary':binary,'decision':classification_summary['decision'],'sanity':len(sanity_rows)})


{'classification_accuracy': 1.0, 'confusion': {'proper_noun': {'proper_noun': 10, 'numeric_condition': 0, 'semantic': 0}, 'numeric_condition': {'proper_noun': 0, 'numeric_condition': 10, 'semantic': 0}, 'semantic': {'proper_noun': 0, 'numeric_condition': 0, 'semantic': 10}}, 'binary': {'tp': 10, 'tn': 20, 'fp': 0, 'fn': 0, 'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'specificity': 1.0, 'false_reranker': 0, 'missed_reranker': 0}, 'decision': 'dev_rule_exactly_reproduces_gold_route', 'sanity': 12}


In [3]:
SOURCE_PAYLOAD=('card_hit_at_3','hit_at_3','recall_at_5','mrr_at_5','ndcg_at_5','relevant_unit_count','ranking_unit_count','top5_chunk_ids','top5_cards','top5_levels')
def payload_digest(row): return hashlib.sha256(json.dumps({k:str(row[k]) for k in SOURCE_PAYLOAD},ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()).hexdigest()
rows18=read_csv(INPUTS['per_query_18']); keys18=[(r['comparison'],r['weight'],int(r['depth']),r['system'],r['query_id'],r['view']) for r in rows18]; assert len(keys18)==len(set(keys18))==1200
lookup18={k:r for k,r in zip(keys18,rows18)}
def source18(weight,depth,system,query_id,view): return lookup18[(f'{weight}_top{depth}',weight,depth,system,query_id,view)]
phase_a_rows=[]
for weight in WEIGHTS:
    for depth in DEPTHS:
        for system in ('all_no','all_gte','gold_oracle','actual_regex'):
            for query_id,item in query_gold.items():
                selected_system='no_reranker' if system=='all_no' else ('gte' if system=='all_gte' else ('gte' if (item['category']=='semantic' if system=='gold_oracle' else prediction[query_id]=='semantic') else 'no_reranker'))
                for view in VIEWS:
                    source=source18(weight,depth,selected_system,query_id,view); key=(f'{weight}_top{depth}',weight,depth,selected_system,query_id,view)
                    phase_a_rows.append({'weight':weight,'depth':depth,'system':system,'query_id':query_id,'question_group':source['question_group'],'category':item['category'],'predicted_category':prediction[query_id],'view':view,'selected_source_system':selected_system,'source_comparison':key[0],'source_weight':key[1],'source_depth':key[2],'source_system':key[3],'source_query_id':key[4],'source_view':key[5],'source_canonical_key':json.dumps(list(key),separators=(',',':')),'source_payload_sha256':payload_digest(source),**{m:float(source[m]) for m in METRICS},'relevant_unit_count':int(source['relevant_unit_count']),'ranking_unit_count':int(source['ranking_unit_count']),'top5_chunk_ids':source['top5_chunk_ids'],'top5_cards':source['top5_cards'],'top5_levels':source['top5_levels']})
assert len(phase_a_rows)==4*4*30*5
def in_group(row,group): return group=='all' or (group=='card' and row['question_group']=='card') or (group=='evidence' and row['question_group']=='evidence') or (group=='numeric' and row['category']=='numeric_condition') or (group=='semantic' and row['category']=='semantic')
phase_a_summary=[]
for weight in WEIGHTS:
    for depth in DEPTHS:
        for system in ('all_no','all_gte','gold_oracle','actual_regex'):
            for view in VIEWS:
                rows=[r for r in phase_a_rows if r['weight']==weight and r['depth']==depth and r['system']==system and r['view']==view]
                for group in GROUPS:
                    selected=[r for r in rows if in_group(r,group)]; assert len(selected)==DENOMS[group]; phase_a_summary.append({'weight':weight,'depth':depth,'system':system,'view':view,'group':group,'denominator':len(selected),**{m:statistics.fmean(r[m] for r in selected) for m in METRICS}})
phase_a_lookup={(r['weight'],r['depth'],r['system'],r['query_id'],r['view']):r for r in phase_a_rows}; phase_a_paired=[]
for weight in WEIGHTS:
    for depth in DEPTHS:
        for reference in ('all_no','gold_oracle'):
            for query_id in query_gold:
                for view in VIEWS:
                    cur=phase_a_lookup[(weight,depth,'actual_regex',query_id,view)]; base=phase_a_lookup[(weight,depth,reference,query_id,view)]; row={'weight':weight,'depth':depth,'reference':reference,'query_id':query_id,'question_group':cur['question_group'],'category':cur['category'],'view':view}
                    for m in METRICS: delta=cur[m]-base[m]; row[f'delta_{m}']=delta; row[f'{m}_outcome']='win' if delta>TOL else ('loss' if delta < -TOL else 'tie')
                    phase_a_paired.append(row)
phase_a_wlt=[]
for weight in WEIGHTS:
    for depth in DEPTHS:
        for reference in ('all_no','gold_oracle'):
            for view in VIEWS:
                for group in GROUPS:
                    rows=[r for r in phase_a_paired if r['weight']==weight and r['depth']==depth and r['reference']==reference and r['view']==view and in_group(r,group)]
                    for m in METRICS:
                        counts={x:sum(r[f'{m}_outcome']==x for r in rows) for x in ('win','loss','tie')}; assert sum(counts.values())==DENOMS[group]; phase_a_wlt.append({'weight':weight,'depth':depth,'reference':reference,'view':view,'group':group,'metric':m,'denominator':len(rows),'wins':counts['win'],'losses':counts['loss'],'ties':counts['tie'],'mean_delta':statistics.fmean(r[f'delta_{m}'] for r in rows)})
assert len(phase_a_summary)==400 and len(phase_a_paired)==1200 and len(phase_a_wlt)==1000
print({'phase_a_rows':len(phase_a_rows),'summary':len(phase_a_summary),'paired':len(phase_a_paired),'wlt':len(phase_a_wlt),'canonical_sources':True})


{'phase_a_rows': 2400, 'summary': 400, 'paired': 1200, 'wlt': 1000, 'canonical_sources': True}


## Phase B — 같은 leaf 후보에서 GTE/BGE × 원문/경로보강

Evidence 20개 질의의 저장 Top50에서 section/benefit만 남긴 동일 후보를 사용한다. 모델 입력에는 원문 query와 허용된 document 또는 issuer/card name/level/첫 heading만 보강한 document가 들어간다.


In [4]:
chunks=read_jsonl(INPUTS['chunks_13']); chunk_by_id={r['id']:r for r in chunks}; assert len(chunks)==len(chunk_by_id)==327
evidence_ids=[q for q,v in query_gold.items() if v['expected_level']!='card']; assert len(evidence_ids)==20
candidate_source=[r for r in read_csv(INPUTS['candidates_16']) if r['configuration'] in WEIGHTS]; pools={}; original_rank={}
for weight in WEIGHTS:
    for query_id in evidence_ids:
        rows=sorted((r for r in candidate_source if r['configuration']==weight and r['query_id']==query_id),key=lambda r:int(r['fused_rank'])); assert len(rows)==50 and [int(r['fused_rank']) for r in rows]==list(range(1,51))
        leaf=[r['chunk_id'] for r in rows if chunk_by_id[r['chunk_id']]['metadata']['level'] in ('section','benefit')]; assert 34<=len(leaf)<=46 and len(set(leaf))==len(leaf)
        pools[(weight,'leaf_only_top20',query_id)]=leaf[:20]; pools[(weight,'leaf_available_from_original_top50',query_id)]=leaf
        original_rank[(weight,query_id)]={r['chunk_id']:int(r['fused_rank']) for r in rows}
pair_keys=sorted({(q,c) for (w,v,q),pool in pools.items() for c in pool}); assert len(pair_keys)==933 and all(q in evidence_ids and chunk_by_id[c]['metadata']['level'] in ('section','benefit') for q,c in pair_keys)
HEADING_RE=re.compile(r'^\s{0,3}#{1,6}\s+(.+?)\s*$',re.MULTILINE)
def augmented_document(chunk):
    meta=chunk['metadata']; match=HEADING_RE.search(chunk['document']); heading=match.group(1).strip() if match else ''; assert not heading or norm(heading) in norm(chunk['document'])
    path=f"{meta['issuer']} > {meta['card_name']} > {meta['level']}"+(f' > {heading}' if heading else '')
    return f'[문서 경로]\n{path}\n\n[본문]\n{chunk["document"]}',heading
augmented={}; heading_rows=[]
for chunk_id in {c for _,c in pair_keys}:
    text,heading=augmented_document(chunk_by_id[chunk_id]); augmented[chunk_id]=text; heading_rows.append({'chunk_id':chunk_id,'heading':heading,'heading_present':bool(heading),'heading_normalized_contained':True,'augmented_sha256':hashlib.sha256(text.encode()).hexdigest()})
forbidden=('section','label_ids','label_kinds','structured_metadata','structured_path','expected_card','required_terms','strict_relevant','coverage_status','source_path','parent_id','page_num','input_fingerprint')
assert all(token not in ('issuer','card_name','level','heading','document') for token in forbidden)
relevance={r['query_id']:r for r in read_jsonl(INPUTS['relevance_18'])}; strict={q:set(r['strict_chunk_ids']) for q,r in relevance.items()}; answer={q:set(r['answer_bearing_chunk_ids']) for q,r in relevance.items()}; strict_groups={q:set(r['strict_exact_group_ids']) for q,r in relevance.items()}; answer_groups={q:set(r['answer_bearing_exact_group_ids']) for q,r in relevance.items()}
group_by_chunk={}
for r in read_jsonl(INPUTS['exact_groups_18']):
    for c in r['member_chunk_ids']: group_by_chunk[c]=r['group_id']
saved_gte={(r['query_id'],r['chunk_id']):float(r['reranker_logit']) for r in read_csv(INPUTS['gte_scores_17'])}; assert all(pair in saved_gte for pair in pair_keys)
leaf18=read_csv(INPUTS['leaf_per_query_18']); print({'evidence_queries':len(evidence_ids),'unique_pairs':len(pair_keys),'pool_k':[min(len(v) for v in pools.values()),max(len(v) for v in pools.values())],'augmented_documents':len(augmented),'no_leak_fields':True})


{'evidence_queries': 20, 'unique_pairs': 933, 'pool_k': [20, 46], 'augmented_documents': 230, 'no_leak_fields': True}


In [5]:
score_cache={}; pair_score_rows=[]; resource_rows=[]
def score_config(model_name,input_variant,tokenizer,model,load_seconds):
    documents={c:(chunk_by_id[c]['document'] if input_variant=='original' else augmented[c]) for _,c in pair_keys}; raw_q={q:len(tokenizer.encode(query_gold[q]['query'],add_special_tokens=False)) for q in evidence_ids}; raw_d={c:len(tokenizer.encode(documents[c],add_special_tokens=False)) for c in documents}; ordered=sorted(pair_keys,key=lambda p:(raw_q[p[0]]+raw_d[p[1]],p)); special=tokenizer.num_special_tokens_to_add(pair=True)
    warm=tokenizer(['워밍업 질문'],['짧은 문서'],padding=True,truncation='only_second',max_length=MAX_LENGTH,return_tensors='pt').to('cuda'); torch.cuda.reset_peak_memory_stats(); started=time.perf_counter()
    with torch.inference_mode(): logits=model(**warm).logits
    torch.cuda.synchronize(); warmup=time.perf_counter()-started; assert logits.shape==(1,1); del warm,logits
    rows=[]; torch.cuda.reset_peak_memory_stats(); started=time.perf_counter()
    try:
        for start in range(0,len(ordered),BATCH_SIZE):
            batch=ordered[start:start+BATCH_SIZE]; encoded=tokenizer([query_gold[q]['query'] for q,c in batch],[documents[c] for q,c in batch],padding=True,truncation='only_second',max_length=MAX_LENGTH,return_tensors='pt'); counts=encoded['attention_mask'].sum(1).tolist(); encoded={k:v.to('cuda') for k,v in encoded.items()}
            with torch.inference_mode(): output=model(**encoded).logits
            assert output.ndim==2 and output.shape==(len(batch),1); values=output[:,0].float().cpu().tolist()
            for (q,c),value,count in zip(batch,values,counts):
                used=max(0,count-raw_q[q]-special); truncated=max(0,raw_d[c]-used); assert math.isfinite(value) and count<=MAX_LENGTH and used<=raw_d[c]
                score_cache[(model_name,input_variant,q,c)]=value; rows.append({'model':model_name,'input_variant':input_variant,'query_id':q,'chunk_id':c,'reranker_logit':value,'level':chunk_by_id[c]['metadata']['level'],'input_document_sha256':hashlib.sha256(documents[c].encode()).hexdigest(),'query_tokens_raw':raw_q[q],'document_tokens_raw':raw_d[c],'input_tokens':count,'document_tokens_input':used,'document_tokens_truncated':truncated,'document_truncation_ratio':truncated/raw_d[c] if raw_d[c] else 0.0,'batch_size':BATCH_SIZE})
            del encoded,output
        torch.cuda.synchronize()
    except torch.cuda.OutOfMemoryError:
        score_cache.clear(); rows.clear(); torch.cuda.empty_cache(); raise RuntimeError(f'OOM {model_name}/{input_variant} at batch2; partial results discarded; fresh-kernel batch1 rerun required')
    seconds=time.perf_counter()-started; alloc=torch.cuda.max_memory_allocated(); reserved=torch.cuda.max_memory_reserved(); assert len(rows)==933
    resource_rows.append({'model':model_name,'input_variant':input_variant,'revision':GTE_REV if model_name=='gte' else BGE_REV,'load_seconds':load_seconds,'warmup_seconds':warmup,'scoring_seconds':seconds,'pairs':len(rows),'pairs_per_second':len(rows)/seconds,'batch_size':BATCH_SIZE,'dtype':'float16','max_length':MAX_LENGTH,'truncation':'only_second','peak_allocated_bytes':alloc,'peak_reserved_bytes':reserved,'input_tokens_p50':statistics.median(r['input_tokens'] for r in rows),'input_tokens_p95':percentile([r['input_tokens'] for r in rows],95),'input_tokens_max':max(r['input_tokens'] for r in rows),'truncated_pairs':sum(r['document_tokens_truncated']>0 for r in rows),'oom':False,'cache_bytes':cache_before[model_name]['total_bytes']}); pair_score_rows.extend(rows)

load_started=time.perf_counter(); gte_tokenizer=AutoTokenizer.from_pretrained(GTE_ROOT,local_files_only=True); gte_model=AutoModelForSequenceClassification.from_pretrained(GTE_ROOT,trust_remote_code=True,local_files_only=True,code_revision=CUSTOM_REV,dtype=torch.float16).eval().to('cuda'); torch.cuda.synchronize(); gte_load=time.perf_counter()-load_started; assert gte_tokenizer.model_max_length==32768
score_config('gte','original',gte_tokenizer,gte_model,gte_load); score_config('gte','augmented',gte_tokenizer,gte_model,0.0); del gte_model,gte_tokenizer; gc.collect(); torch.cuda.empty_cache()
load_started=time.perf_counter(); bge_tokenizer=AutoTokenizer.from_pretrained(BGE_ROOT,local_files_only=True,trust_remote_code=False); bge_model=AutoModelForSequenceClassification.from_pretrained(BGE_ROOT,local_files_only=True,trust_remote_code=False,dtype=torch.float16).eval().to('cuda'); torch.cuda.synchronize(); bge_load=time.perf_counter()-load_started; assert bge_tokenizer.model_max_length==8192 and bge_model.__class__.__name__=='XLMRobertaForSequenceClassification'
score_config('bge','original',bge_tokenizer,bge_model,bge_load); score_config('bge','augmented',bge_tokenizer,bge_model,0.0); del bge_model,bge_tokenizer; gc.collect(); torch.cuda.empty_cache()
assert len(score_cache)==4*933 and len(pair_score_rows)==4*933
gte_diffs=[abs(score_cache[('gte','original',q,c)]-saved_gte[(q,c)]) for q,c in pair_keys]; assert all(math.isfinite(x) for x in gte_diffs) and max(gte_diffs)<=GTE_REPRO_TOL
gte_repro_pools=[]
for (weight,variant,q),pool in pools.items():
    new=sorted(pool,key=lambda c:(-score_cache[('gte','original',q,c)],original_rank[(weight,q)][c],c)); old=sorted(pool,key=lambda c:(-saved_gte[(q,c)],original_rank[(weight,q)][c],c)); pos_new={c:i for i,c in enumerate(new)}; pos_old={c:i for i,c in enumerate(old)}; rho=float(np.corrcoef([pos_new[c] for c in pool],[pos_old[c] for c in pool])[0,1]); gte_repro_pools.append({'weight':weight,'variant':variant,'query_id':q,'full_rank_exact':new==old,'top5_exact':new[:5]==old[:5],'rank_correlation':rho})
assert len(gte_repro_pools)==80 and all(r['full_rank_exact'] and r['top5_exact'] and r['rank_correlation']>=1-1e-12 for r in gte_repro_pools)
old_score_rows=read_csv(INPUTS['gte_scores_17']); old_order=sorted(old_score_rows,key=lambda r:(int(r['query_tokens_raw'])+int(r['document_tokens_raw']),(r['query_id'],r['chunk_id']))); new_gte_rows=sorted((r for r in pair_score_rows if r['model']=='gte' and r['input_variant']=='original'),key=lambda r:(int(r['query_tokens_raw'])+int(r['document_tokens_raw']),(r['query_id'],r['chunk_id'])))
def batch_padding_context(rows):
    result={}
    for i in range(0,len(rows),BATCH_SIZE):
        batch=rows[i:i+BATCH_SIZE]; padded=max(int(r['input_tokens']) for r in batch)
        for r in batch: result[(r['query_id'],r['chunk_id'])]=padded
    return result
old_context=batch_padding_context(old_order); new_context=batch_padding_context(new_gte_rows); same_context=[p for p in pair_keys if old_context[p]==new_context[p]]; changed_context=[p for p in pair_keys if old_context[p]!=new_context[p]]
gte_repro={'pairs':933,'diagnostic_logit_abs_tolerance':GTE_REPRO_TOL,'abs_diff_median':statistics.median(gte_diffs),'abs_diff_p95':percentile(gte_diffs,95),'abs_diff_p99':percentile(gte_diffs,99),'max_abs_diff':max(gte_diffs),'exact_count':sum(x==0 for x in gte_diffs),'finite_count':sum(math.isfinite(x) for x in gte_diffs),'same_padding_context_pairs':len(same_context),'same_padding_context_mean_abs_diff':statistics.fmean(abs(score_cache[('gte','original',q,c)]-saved_gte[(q,c)]) for q,c in same_context),'changed_padding_context_pairs':len(changed_context),'changed_padding_context_mean_abs_diff':statistics.fmean(abs(score_cache[('gte','original',q,c)]-saved_gte[(q,c)]) for q,c in changed_context),'full_rank_exact_pools':sum(r['full_rank_exact'] for r in gte_repro_pools),'top5_exact_pools':sum(r['top5_exact'] for r in gte_repro_pools),'rank_correlation_min':min(r['rank_correlation'] for r in gte_repro_pools),'quality_contract':'logit tolerance is diagnostic only; full-rank, Top5, and 18 metrics exact are hard assertions'}
print({'scored_pairs':len(pair_score_rows),'gte_saved_reproduction':gte_repro,'resources':resource_rows})


{'scored_pairs': 3732, 'gte_saved_reproduction': {'pairs': 933, 'diagnostic_logit_abs_tolerance': 0.002, 'abs_diff_median': 0.0, 'abs_diff_p95': 0.0, 'abs_diff_p99': 0.0, 'max_abs_diff': 0.001708984375, 'exact_count': 929, 'finite_count': 933, 'same_padding_context_pairs': 720, 'same_padding_context_mean_abs_diff': 0.0, 'changed_padding_context_pairs': 213, 'changed_padding_context_mean_abs_diff': 1.719300176056338e-05, 'full_rank_exact_pools': 80, 'top5_exact_pools': 80, 'rank_correlation_min': 0.9999999999999999, 'quality_contract': 'logit tolerance is diagnostic only; full-rank, Top5, and 18 metrics exact are hard assertions'}, 'resources': [{'model': 'gte', 'input_variant': 'original', 'revision': '8215cf04918ba6f7b6a62bb44238ce2953d8831c', 'load_seconds': 0.9053802669513971, 'warmup_seconds': 0.22821622085757554, 'scoring_seconds': 3.732710298150778, 'pairs': 933, 'pairs_per_second': 249.9524274525718, 'batch_size': 2, 'dtype': 'float16', 'max_length': 8192, 'truncation': 'only_se

In [6]:
RERANKERS=('gte_original','gte_augmented','bge_original','bge_augmented'); SYSTEMS=('no_reranker',)+RERANKERS
rankings={}
for weight in WEIGHTS:
    for variant in VARIANTS:
        for q in evidence_ids:
            pool=pools[(weight,variant,q)]; rankings[(weight,variant,'no_reranker',q)]=pool
            for system in RERANKERS:
                model,input_variant=system.split('_',1); ranking=sorted(pool,key=lambda c:(-score_cache[(model,input_variant,q,c)],original_rank[(weight,q)][c],c)); assert len(ranking)==len(set(ranking))==len(pool) and set(ranking)==set(pool); rankings[(weight,variant,system,q)]=ranking
def compressed(raw): return list(dict.fromkeys(group_by_chunk[c] for c in raw))
def view_metrics(q,raw,view):
    card=int(any(chunk_by_id[c]['metadata']['card_key']==query_gold[q]['expected_card'] for c in raw[:3]))
    if view=='strict_raw': units,rel=raw,strict[q]
    elif view=='answer_bearing_raw': units,rel=raw,answer[q]
    elif view=='strict_exact_doc_dedup': units,rel=compressed(raw),strict_groups[q]
    elif view=='answer_bearing_exact_doc_dedup': units,rel=compressed(raw),answer_groups[q]
    else: units,rel=raw,{f'gold_family:{q}'}
    if view=='answer_bearing_gold_family_oracle':
        seen=False; hits=[]
        for c in units[:5]: hit=c in answer[q] and not seen; hits.append(hit); seen=seen or hit
    else: hits=[x in rel for x in units[:5]]
    first=next((i for i,x in enumerate(hits,1) if x),None); dcg=sum(x/math.log2(i+1) for i,x in enumerate(hits,1)); ideal=sum(1/math.log2(i+1) for i in range(1,min(5,len(rel))+1)); result={'card_hit_at_3':card,'hit_at_3':int(any(hits[:3])),'recall_at_5':sum(hits)/len(rel),'mrr_at_5':1/first if first else 0.0,'ndcg_at_5':dcg/ideal,'relevant_unit_count':len(rel),'ranking_unit_count':len(units)}; assert all(0<=result[m]<=1 for m in METRICS); return result
leaf_per_query=[]
for weight in WEIGHTS:
    for variant in VARIANTS:
        for system in SYSTEMS:
            for q in evidence_ids:
                raw=rankings[(weight,variant,system,q)]; by_view={v:view_metrics(q,raw,v) for v in VIEWS}; assert len({by_view[v]['card_hit_at_3'] for v in VIEWS})==1
                for view,result in by_view.items(): leaf_per_query.append({'weight':weight,'variant':variant,'system':system,'query_id':q,'category':query_gold[q]['category'],'view':view,'candidate_count':len(raw),**result,'top5_chunk_ids':json.dumps(raw[:5],separators=(',',':')),'top5_levels':json.dumps([chunk_by_id[c]['metadata']['level'] for c in raw[:5]],separators=(',',':'))})
assert len(leaf_per_query)==2000
leaf18_lookup={(r['weight'], 'leaf_available_from_original_top50' if r['variant']=='leaf_available_from_top50' else r['variant'],r['system'],r['query_id'],r['view']):r for r in leaf18}
for row in leaf_per_query:
    if row['system'] in ('no_reranker','gte_original'):
        old='gte' if row['system']=='gte_original' else 'no_reranker'; source=leaf18_lookup[(row['weight'],row['variant'],old,row['query_id'],row['view'])]; assert row['top5_chunk_ids']==source['top5_chunk_ids'] and row['top5_levels']==source['top5_levels'] and int(row['ranking_unit_count'])==int(source['ranking_unit_count']) and all(abs(float(row[m])-float(source[m]))<=TOL for m in METRICS)
leaf_summary=[]
for weight in WEIGHTS:
    for variant in VARIANTS:
        for system in SYSTEMS:
            for view in VIEWS:
                rows=[r for r in leaf_per_query if r['weight']==weight and r['variant']==variant and r['system']==system and r['view']==view]
                for group in EVIDENCE_GROUPS:
                    selected=[r for r in rows if group=='evidence' or (group=='numeric' and r['category']=='numeric_condition') or (group=='semantic' and r['category']=='semantic')]; expected=20 if group=='evidence' else 10; assert len(selected)==expected; leaf_summary.append({'weight':weight,'variant':variant,'system':system,'view':view,'group':group,'denominator':len(selected),**{m:statistics.fmean(r[m] for r in selected) for m in METRICS}})
leaf_lookup={(r['weight'],r['variant'],r['system'],r['query_id'],r['view']):r for r in leaf_per_query}; leaf_paired=[]
for weight in WEIGHTS:
    for variant in VARIANTS:
        for system in RERANKERS:
            for q in evidence_ids:
                for view in VIEWS:
                    cur=leaf_lookup[(weight,variant,system,q,view)]; base=leaf_lookup[(weight,variant,'no_reranker',q,view)]; row={'weight':weight,'variant':variant,'system':system,'query_id':q,'category':cur['category'],'view':view}
                    for m in METRICS: delta=cur[m]-base[m]; row[f'delta_{m}']=delta; row[f'{m}_outcome']='win' if delta>TOL else ('loss' if delta < -TOL else 'tie')
                    leaf_paired.append(row)
leaf_wlt=[]
for weight in WEIGHTS:
    for variant in VARIANTS:
        for system in RERANKERS:
            for view in VIEWS:
                for group in EVIDENCE_GROUPS:
                    rows=[r for r in leaf_paired if r['weight']==weight and r['variant']==variant and r['system']==system and r['view']==view and (group=='evidence' or (group=='numeric' and r['category']=='numeric_condition') or (group=='semantic' and r['category']=='semantic'))]
                    for m in METRICS:
                        counts={x:sum(r[f'{m}_outcome']==x for r in rows) for x in ('win','loss','tie')}; assert sum(counts.values())==(20 if group=='evidence' else 10); leaf_wlt.append({'weight':weight,'variant':variant,'system':system,'view':view,'group':group,'metric':m,'denominator':len(rows),'wins':counts['win'],'losses':counts['loss'],'ties':counts['tie'],'mean_delta':statistics.fmean(r[f'delta_{m}'] for r in rows)})
assert len(leaf_summary)==300 and len(leaf_paired)==1600 and len(leaf_wlt)==1200
print({'leaf_per_query':len(leaf_per_query),'summary':len(leaf_summary),'paired':len(leaf_paired),'wlt':len(leaf_wlt),'18_leaf_baseline_gte_exact':True})


{'leaf_per_query': 2000, 'summary': 300, 'paired': 1600, 'wlt': 1200, '18_leaf_baseline_gte_exact': True}


In [7]:
selective_rows=[]; regex_gap=[]
for weight in WEIGHTS:
    for variant in VARIANTS:
        for reranker in RERANKERS:
            for q in evidence_ids:
                for view in VIEWS:
                    actual_source=reranker if prediction[q]=='semantic' else 'no_reranker'; gold_source=reranker if query_gold[q]['category']=='semantic' else 'no_reranker'
                    actual=leaf_lookup[(weight,variant,actual_source,q,view)]; gold=leaf_lookup[(weight,variant,gold_source,q,view)]
                    for route,source,source_name in (('actual_regex',actual,actual_source),('gold_category_oracle',gold,gold_source)): selective_rows.append({'weight':weight,'variant':variant,'reranker':reranker,'route':route,'selected_source':source_name,'query_id':q,'category':query_gold[q]['category'],'predicted_category':prediction[q],'view':view,**{m:source[m] for m in METRICS},'top5_chunk_ids':source['top5_chunk_ids']})
                    gap={'weight':weight,'variant':variant,'reranker':reranker,'query_id':q,'category':query_gold[q]['category'],'predicted_category':prediction[q],'view':view,'same_route':actual_source==gold_source,'same_top5':actual['top5_chunk_ids']==gold['top5_chunk_ids']}
                    for m in METRICS: gap[f'delta_{m}']=actual[m]-gold[m]
                    regex_gap.append(gap)
assert len(selective_rows)==3200 and len(regex_gap)==1600
summary_lookup={(r['weight'],r['variant'],r['system'],r['view'],r['group']):r for r in leaf_summary}; primary_key=('vector_0.4_bm25_0.6','leaf_available_from_original_top50'); base=lambda view,metric:summary_lookup[(primary_key[0],primary_key[1],'no_reranker',view,'evidence')][metric]
guard_specs=(('strict_exact_doc_dedup','hit_at_3'),('strict_exact_doc_dedup','recall_at_5'),('strict_exact_doc_dedup','ndcg_at_5'),('strict_raw','hit_at_3'),('strict_raw','mrr_at_5'),('strict_raw','recall_at_5'),('strict_raw','ndcg_at_5'),('strict_raw','card_hit_at_3'),('answer_bearing_exact_doc_dedup','hit_at_3'),('answer_bearing_exact_doc_dedup','mrr_at_5'))
qualifications={}
for system in RERANKERS:
    current=lambda view,metric:summary_lookup[(primary_key[0],primary_key[1],system,view,'evidence')][metric]; primary_delta=current('strict_exact_doc_dedup','mrr_at_5')-base('strict_exact_doc_dedup','mrr_at_5'); guards={f'{view}:{metric}':current(view,metric)+TOL>=base(view,metric) for view,metric in guard_specs}; qualifications[system]={'primary_delta':primary_delta,'primary_improved':primary_delta>TOL,'guardrails':guards,'dev_signal':primary_delta>TOL and all(guards.values())}
factorial=[]
for view,metric in (('strict_exact_doc_dedup','mrr_at_5'),('strict_exact_doc_dedup','ndcg_at_5'),('strict_raw','mrr_at_5'),('answer_bearing_exact_doc_dedup','mrr_at_5')):
    value=lambda s:summary_lookup[(primary_key[0],primary_key[1],s,view,'evidence')][metric]; gte_aug=value('gte_augmented')-value('gte_original'); bge_aug=value('bge_augmented')-value('bge_original'); factorial.append({'view':view,'metric':metric,'gte_augmentation_effect':gte_aug,'bge_augmentation_effect':bge_aug,'bge_vs_gte_original':value('bge_original')-value('gte_original'),'bge_vs_gte_augmented':value('bge_augmented')-value('gte_augmented'),'model_input_interaction':bge_aug-gte_aug})
raw_best=max(RERANKERS,key=lambda s:summary_lookup[(primary_key[0],primary_key[1],s,'strict_exact_doc_dedup','evidence')]['mrr_at_5']); decision={'phase_a':classification_summary,'phase_b':{'primary':{'weight':primary_key[0],'variant':primary_key[1],'view':'strict_exact_doc_dedup','metric':'mrr_at_5'},'qualifications':qualifications,'raw_best':raw_best,'factorial_effects':factorial,'promotion':'not_eligible_for_promotion','selective_regex_gold_gap_rows_nonzero':sum(any(abs(r[f'delta_{m}'])>TOL for m in METRICS) for r in regex_gap)},'final':'current_chunking_prebranch_diagnostic_only'}
cache_after={name:tree_snapshot(path) for name,path in {'gte':GTE_ROOT,'bge':BGE_ROOT,'custom_hub':CUSTOM_HUB,'custom_module':CUSTOM_MODULE}.items()}; assert cache_after==cache_before
source_after={name:sha(path) for name,path in INPUTS.items()}; assert source_after==source_before
resources={'physical_gpu':0,'visible_gpu':0,'gpu_name':torch.cuda.get_device_name(0),'package_versions':versions,'pip_check_preexisting_warning':CONTRACT['execution']['pip_check_preexisting_warning'],'configs':resource_rows,'gte_original_saved_reproduction':gte_repro,'cache_before':cache_before,'cache_after':cache_after,'cache_unchanged':True}
write_csv(OUTPUT_ROOT/'phase_a_classification.csv',classification); write_csv(OUTPUT_ROOT/'phase_a_sanity.csv',sanity_rows); write_json(OUTPUT_ROOT/'phase_a_classification_summary.json',classification_summary); write_csv(OUTPUT_ROOT/'phase_a_per_query_metrics.csv',phase_a_rows); write_csv(OUTPUT_ROOT/'phase_a_summary.csv',phase_a_summary); write_csv(OUTPUT_ROOT/'phase_a_paired_deltas.csv',phase_a_paired); write_csv(OUTPUT_ROOT/'phase_a_wlt.csv',phase_a_wlt)
write_csv(OUTPUT_ROOT/'leaf_pair_scores.csv',pair_score_rows); write_csv(OUTPUT_ROOT/'leaf_per_query_metrics.csv',leaf_per_query); write_csv(OUTPUT_ROOT/'leaf_summary.csv',leaf_summary); write_csv(OUTPUT_ROOT/'leaf_paired_deltas.csv',leaf_paired); write_csv(OUTPUT_ROOT/'leaf_wlt.csv',leaf_wlt); write_csv(OUTPUT_ROOT/'leaf_selective_routes.csv',selective_rows); write_csv(OUTPUT_ROOT/'leaf_regex_gold_gap.csv',regex_gap); write_csv(OUTPUT_ROOT/'augmented_heading_audit.csv',heading_rows); write_json(OUTPUT_ROOT/'resources.json',resources); write_json(OUTPUT_ROOT/'decision.json',decision)
summary_json={'schema_version':'current_chunking_prebranch_summary_v1','phase_a':{'classification':classification_summary,'summary_rows':phase_a_summary},'phase_b':{'summary_rows':leaf_summary,'decision':decision['phase_b']},'resources':resources,'limitations':['development queries only','gold oracle is not deployable','regex exactness on 30 queries does not validate unseen wording','rerankers cannot recover outside stored leaf candidates','raw GTE/BGE logit magnitudes are not compared']}
write_json(OUTPUT_ROOT/'summary.json',summary_json)
readme=f'''# 21 현재 청킹 분기 전 종합 평가

- actual regex route: 정답을 보지 않고 질문 문구만 보는 실제 규칙
- selective oracle: 정답 유형을 미리 아는 상한이며 운영 후보가 아님
- Precision(정밀도): 해당 유형이라고 예측한 것 중 맞은 비율. 예: numeric 예측의 정확성
- Recall(재현율): 실제 해당 유형 중 찾아낸 비율. 표현이 바뀌면 낮아질 수 있음
- F1: Precision과 Recall의 조화평균. 작은 개발셋에서는 변동이 큼
- Specificity(특이도): rerank하지 않아야 할 질문을 passthrough한 비율
- Hit@3: 상위 3개 안에 관련 청크가 있는 비율
- Recall@5: 전체 관련 청크 중 상위 5개가 회수한 비율
- MRR@5: 첫 관련 청크가 얼마나 앞에 있는지 나타내는 평균
- nDCG@5: 관련 청크의 상위 순서 품질. 관련 청크 수에 민감함

Phase A 분류 판정: `{classification_summary['decision']}`. 항상 development_rule_only/not_validated_for_unseen_queries/not_eligible_for_promotion이다.

Phase B primary raw best: `{raw_best}`. 모델·augmentation 효과는 decision.json의 factorial_effects에 분리했다. GTE와 BGE logit 절대값은 비교하지 않는다.

저장된 17번 GTE logit 재현에서 0.002는 fp16 배치 padding 차이를 확인하는 수치 진단 한계일 뿐이다. 품질 계약은 80개 후보 pool의 전체 순위와 Top5, 그리고 18번 지표를 exact 재현하는 것이며 모두 hard assertion이다.

개발셋 단일 실행이며 현재 청킹과 저장 후보에 한정된다. GPU physical 0을 사용했고 network/API/new embedding/Chroma/package install은 0이다. 기존 pip check의 torch-setuptools 경고는 수정하지 않았다.
'''; (OUTPUT_ROOT/'README.md').write_text(readme,encoding='utf-8')
output_names=('evaluation_contract.json','phase_a_classification.csv','phase_a_sanity.csv','phase_a_classification_summary.json','phase_a_per_query_metrics.csv','phase_a_summary.csv','phase_a_paired_deltas.csv','phase_a_wlt.csv','leaf_pair_scores.csv','leaf_per_query_metrics.csv','leaf_summary.csv','leaf_paired_deltas.csv','leaf_wlt.csv','leaf_selective_routes.csv','leaf_regex_gold_gap.csv','augmented_heading_audit.csv','resources.json','decision.json','summary.json','README.md')
integrity={'schema_version':'current_chunking_prebranch_integrity_v1','self_hash_excluded':True,'inputs':{n:{'path':str(p.relative_to(PROJECT_ROOT)),'before':source_before[n],'after':source_after[n],'unchanged':True} for n,p in INPUTS.items()},'cache_before':cache_before,'cache_after':cache_after,'cache_unchanged':True,'outputs':{n:{'sha256':sha(OUTPUT_ROOT/n),'bytes':(OUTPUT_ROOT/n).stat().st_size} for n in output_names},'notebook':{'sha256':'pending_after_nbclient_serialization'},'row_counts':{'phase_a_classification':len(classification),'phase_a_per_query':len(phase_a_rows),'phase_a_summary':len(phase_a_summary),'phase_a_paired':len(phase_a_paired),'phase_a_wlt':len(phase_a_wlt),'pair_scores':len(pair_score_rows),'leaf_per_query':len(leaf_per_query),'leaf_summary':len(leaf_summary),'leaf_paired':len(leaf_paired),'leaf_wlt':len(leaf_wlt),'leaf_selective':len(selective_rows),'regex_gap':len(regex_gap)},'assertions':{'rule_hash_frozen':True,'classification_runtime_query_only':True,'sanity_pass':True,'canonical_source_exact':True,'pair_count_933':True,'four_configs_rescored':True,'no_leak':True,'permutation':True,'18_leaf_baseline_gte_exact':True,'gte_saved_logit_within_diagnostic_tolerance':True,'gte_saved_full_rank_exact_80_of_80':True,'gte_saved_top5_exact_80_of_80':True,'metrics_range_card_invariant':True,'source_cache_unchanged':True},'execution':CONTRACT['execution']}
write_json(OUTPUT_ROOT/'integrity.json',integrity)
print({'phase_a':{'accuracy':accuracy,'decision':classification_summary['decision']},'phase_b':{'raw_best':raw_best,'qualifications':qualifications,'regex_gap_nonzero':decision['phase_b']['selective_regex_gold_gap_rows_nonzero']},'rows':integrity['row_counts'],'source_cache_unchanged':True})


{'phase_a': {'accuracy': 1.0, 'decision': 'dev_rule_exactly_reproduces_gold_route'}, 'phase_b': {'raw_best': 'bge_augmented', 'qualifications': {'gte_original': {'primary_delta': -0.016666666666666607, 'primary_improved': False, 'guardrails': {'strict_exact_doc_dedup:hit_at_3': True, 'strict_exact_doc_dedup:recall_at_5': True, 'strict_exact_doc_dedup:ndcg_at_5': True, 'strict_raw:hit_at_3': True, 'strict_raw:mrr_at_5': False, 'strict_raw:recall_at_5': True, 'strict_raw:ndcg_at_5': False, 'strict_raw:card_hit_at_3': True, 'answer_bearing_exact_doc_dedup:hit_at_3': True, 'answer_bearing_exact_doc_dedup:mrr_at_5': False}, 'dev_signal': False}, 'gte_augmented': {'primary_delta': 0.02416666666666667, 'primary_improved': True, 'guardrails': {'strict_exact_doc_dedup:hit_at_3': True, 'strict_exact_doc_dedup:recall_at_5': True, 'strict_exact_doc_dedup:ndcg_at_5': True, 'strict_raw:hit_at_3': True, 'strict_raw:mrr_at_5': False, 'strict_raw:recall_at_5': True, 'strict_raw:ndcg_at_5': True, 'stric

In [8]:
expected={'phase_a_classification.csv':30,'phase_a_sanity.csv':12,'phase_a_per_query_metrics.csv':2400,'phase_a_summary.csv':400,'phase_a_paired_deltas.csv':1200,'phase_a_wlt.csv':1000,'leaf_pair_scores.csv':3732,'leaf_per_query_metrics.csv':2000,'leaf_summary.csv':300,'leaf_paired_deltas.csv':1600,'leaf_wlt.csv':1200,'leaf_selective_routes.csv':3200,'leaf_regex_gold_gap.csv':1600}
for name,count in expected.items(): assert len(read_csv(OUTPUT_ROOT/name))==count
stored_scores=read_csv(OUTPUT_ROOT/'leaf_pair_scores.csv'); assert len({(r['model'],r['input_variant'],r['query_id'],r['chunk_id']) for r in stored_scores})==3732 and all(math.isfinite(float(r['reranker_logit'])) and 0<=float(r['document_truncation_ratio'])<=1 for r in stored_scores)
stored_a=read_csv(OUTPUT_ROOT/'phase_a_per_query_metrics.csv'); stored_b=read_csv(OUTPUT_ROOT/'leaf_per_query_metrics.csv'); assert all(0<=float(r[m])<=1 for r in stored_a+stored_b for m in METRICS)
assert all(int(r['wins'])+int(r['losses'])+int(r['ties'])==int(r['denominator']) for r in read_csv(OUTPUT_ROOT/'phase_a_wlt.csv')+read_csv(OUTPUT_ROOT/'leaf_wlt.csv'))
assert {n:sha(p) for n,p in INPUTS.items()}==source_before and {name:tree_snapshot(path) for name,path in {'gte':GTE_ROOT,'bge':BGE_ROOT,'custom_hub':CUSTOM_HUB,'custom_module':CUSTOM_MODULE}.items()}==cache_before
stored_integrity=json.loads((OUTPUT_ROOT/'integrity.json').read_text()); assert stored_integrity['cache_unchanged'] and all(v['unchanged'] for v in stored_integrity['inputs'].values())
print({'validation':'PASS','rows':expected,'classification_accuracy':accuracy,'unique_pairs':933,'four_score_configs':True,'source_cache_unchanged':True,'gpu':torch.cuda.get_device_name(0),'network_api_embedding_chroma_install':0})


{'validation': 'PASS', 'rows': {'phase_a_classification.csv': 30, 'phase_a_sanity.csv': 12, 'phase_a_per_query_metrics.csv': 2400, 'phase_a_summary.csv': 400, 'phase_a_paired_deltas.csv': 1200, 'phase_a_wlt.csv': 1000, 'leaf_pair_scores.csv': 3732, 'leaf_per_query_metrics.csv': 2000, 'leaf_summary.csv': 300, 'leaf_paired_deltas.csv': 1600, 'leaf_wlt.csv': 1200, 'leaf_selective_routes.csv': 3200, 'leaf_regex_gold_gap.csv': 1600}, 'classification_accuracy': 1.0, 'unique_pairs': 933, 'four_score_configs': True, 'source_cache_unchanged': True, 'gpu': 'NVIDIA GeForce RTX 3090', 'network_api_embedding_chroma_install': 0}
